# FinAssist Track A: full pipeline on a Colab GPU

Runs every stage (baselines, 3 QLoRA runs, RAG, evaluation, DPO, merge, Q4_K_M quantization,
serving, red-team, gateway) with **Qwen2.5-7B-Instruct** on the `notebook` compute profile.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (or better).
2. *Secrets* (key icon on the left), enable notebook access for each:
   - `WANDB_API_KEY`: optional; without it W&B logs offline.
   - `JUDGE_API_KEY`: optional; needed for the LLM-judge scores.
   - `HF_TOKEN`: only needed for Llama models.
   - `GITHUB_TOKEN`: only if the repo is private.
3. Run all cells. Results land in Google Drive (`MyDrive/finassist_track_a/outputs`).

**Resumable.** A free Colab session stops after a few hours. Rerun all cells: finished
stages are skipped because their outputs are on Drive. Expect several sessions on a T4
(training is roughly 3-6 h per run); Kaggle (30 GPU-h/week) or the IIT box is faster.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# Code in /content (fast disk), outputs on Google Drive (survive disconnects).
import json
import os, subprocess
from google.colab import drive, userdata
drive.mount('/content/drive')

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

REPO = "sparsh610/finassist-track-a"
token = secret("GITHUB_TOKEN")
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if os.path.isdir("/content/track_a/.git"):
    subprocess.run(["git", "-C", "/content/track_a", "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", url, "/content/track_a"], check=True)

DRIVE_OUT = "/content/drive/MyDrive/finassist_track_a/outputs"
os.makedirs(DRIVE_OUT, exist_ok=True)
os.chdir("/content/track_a")
if not os.path.islink("outputs"):
    subprocess.run(["rm", "-rf", "outputs"])
    os.symlink(DRIVE_OUT, "outputs")
print("outputs ->", os.path.realpath("outputs"))

In [ ]:
%%capture install_log
# Model stack (Colab already ships torch with CUDA).
!pip install -q -r requirements-gpu.txt && pip install -q -e . && pip install -q sentencepiece

In [ ]:
# Secrets -> environment for the pipeline.
for name in ("WANDB_API_KEY", "JUDGE_API_KEY", "HF_TOKEN"):
    value = secret(name)
    if value:
        os.environ[name] = value
if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_MODE"] = "offline"   # never block on a W&B login prompt
print({n: bool(os.environ.get(n)) for n in ("WANDB_API_KEY", "JUDGE_API_KEY", "HF_TOKEN")})

In [ ]:
# Serving stack for the Week 4 stages: llama.cpp (GGUF convert + llama-quantize) and Ollama.
import glob, shutil, urllib.request, zipfile
if not os.path.exists("llama.cpp/build/bin/llama-quantize"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", "llama.cpp"], check=True)
    rel = json.load(urllib.request.urlopen("https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=10"))
    asset = next(a["browser_download_url"] for r in rel for a in r["assets"]
                 if a["name"].endswith("bin-ubuntu-x64.zip"))
    urllib.request.urlretrieve(asset, "/tmp/llama-bin.zip")
    zipfile.ZipFile("/tmp/llama-bin.zip").extractall("/tmp/llama-bin")
    qbin = glob.glob("/tmp/llama-bin/**/llama-quantize", recursive=True)[0]
    os.makedirs("llama.cpp/build/bin", exist_ok=True)
    for f in glob.glob(os.path.join(os.path.dirname(qbin), "*")):
        shutil.copy2(f, "llama.cpp/build/bin/")
    os.chmod("llama.cpp/build/bin/llama-quantize", 0o755)
print("llama-quantize:", os.path.exists("llama.cpp/build/bin/llama-quantize"))

if shutil.which("ollama") is None:
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)

In [ ]:
# Start the pipeline in the background (stages run one at a time, finished ones are skipped).
env = dict(os.environ, COMPUTE="notebook", SERVE="ollama")
pipeline = subprocess.Popen(["bash", "scripts/run_pipeline.sh"], env=env)
print("pipeline started, pid", pipeline.pid)

In [ ]:
# Follow progress (safe to stop and rerun this cell; the pipeline keeps running).
import time
from IPython.display import clear_output
while True:
    clear_output(wait=True)
    print(open("outputs/logs/pipeline.log").read()[-4000:])
    if pipeline.poll() is not None:
        print("pipeline exited with code", pipeline.returncode)
        break
    time.sleep(60)

In [ ]:
# Results
import pandas as pd
board = pd.read_csv("outputs/eval/leaderboard.csv")
display(board[["label", "n", "exact_match", "f1", "schema_validity", "latency_p50_s", "judge_correctness"]])
print(open("outputs/eval/retrieval_recall_notebook.json").read() if os.path.exists("outputs/eval/retrieval_recall_notebook.json") else "")
if os.path.exists("docs/redteam_log.md"):
    shutil.copy2("docs/redteam_log.md", "outputs/redteam_log.md")   # keep a copy on Drive
    print(open("docs/redteam_log.md").read()[-1500:])

If a stage failed, its log is in `outputs/logs/<stage>.log`. Fix the cause, then rerun the
pipeline cell: only unfinished stages run again.